In [1]:
#packages
import pandas as pd
from surprise import Dataset, Reader, SVD


In [2]:
#load the clean data
ratings = pd.read_csv('ratings_clean.csv')
recipes = pd.read_csv('recipes_clean.csv')


reader = Reader(rating_scale=(1,5))
data = Dataset.load_from_df(
    ratings[['user_id', 'recipe_id', 'rating']],
    reader
)


In [3]:
svd = SVD(random_state=1)
trainset = data.build_full_trainset()
svd.fit(trainset)

In [4]:
#without ingredients, only using CF
def recommender(userId):
    #removing recipes that the user has already rated
    rated = ratings[ratings['user_id'] == userId]['recipe_id']
    recipes_r = recipes[~recipes['id'].isin(rated)].copy()

    #prediction rating for each recipe, clip=false to lets values go above 5
    recipes_r['est'] = recipes_r['id'].apply(lambda x: svd.predict(userId, x, clip=False).est)

    #sort prediction, and returns top 10
    recipes_r = recipes_r.sort_values('est', ascending=False)

    return recipes_r[['name', 'est']].head(10)




In [ ]:
#Recommmender 2: cascade hybrid
def recommender_ingredients(userId, my_ingredients):
    #count amount of users ingredients each recipe contains
    match_scores = recipes['ingredients'].apply(lambda x: sum(i.strip() in x for i in my_ingredients.split(',') if i.strip()))

    #sort them by number of matches
    sim_scores = list(enumerate(match_scores))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    #stops if no recipe contains any of the ingredients
    if sim_scores[0][1] == 0:
        return "No recipes contain any of these ingredients"

    #Only keeps recipes with at least one match
    sim_scores = [i for i in sim_scores if i[1]>0]

    recipe_indices = [i[0] for i in sim_scores]

    recipes_c = recipes.iloc[recipe_indices][['name', 'n_ingredients', 'id']]

    recipes_c['matches'] = [i[1] for i in sim_scores]

    #predict rating for each candidate with SVD
    recipes_c['est'] = recipes_c['id'].apply(lambda x: svd.predict(userId, x, clip=False).est)

    #sort by most matches, then fewest total ingredients, then highest predicted rating
    recipes_c = recipes_c.sort_values(['matches', 'n_ingredients', 'est'], ascending=[False, True, False])

    return recipes_c.head(10)

In [ ]:
#example of first recommender, only using CF
recommender(1701315) #other random user ids: 101823 868654


,name,est
4851,fresh tomato and cucumber salad,5.284950
6342,jeni s chocolate ganache,5.271098
1494,blue ribbon carrot cake with buttermilk glaze,5.269738
9243,pumpkin ginger scones with cinnamon chips,5.269339
5127,globally inspired fully loaded rolled mashed ...,5.266752
8509,paige s buttercream frosting,5.264726
83,2 handed kitchen sink tomato sandwich,5.258429
2687,chipotle mexican grill barbacoa burritos by to...,5.257164
1644,bread and butter pickles my way,5.254889
1573,boneless breast of chicken saltimbocca,5.254444


In [ ]:
#example of second recommender, with ingredients the user has at home
recommender_ingredients(1701315, 'red onion, potato, pear, lemon, lime, beans, egg, lettuce, garlic,')

,name,n_ingredients,id,matches,est
5026,garlicky cilantro roasted potato salad,10,31286,5,4.921074
7508,mexican chef salad,16,144969,5,5.023206
12388,youvarlakia avgolemono greek meatball egg lem...,17,59197,5,4.878390
6935,lemony green beans,7,201257,4,4.832455
6788,lebanese green bean salad,8,172357,4,4.690262
3992,dilled white bean and grape tomato salad,10,428248,4,5.035656
7768,moosewood white bean and tomato salad,10,177902,4,5.027129
9731,roasted potatoes and green beans,10,107740,4,5.015406
1424,black bean and corn salad,10,39572,4,4.940683
10579,spanish green bean salad,11,229473,4,5.027531


: 